# Exploration notebook (legacy)

The original exploration that led to this project. Outputs are cleared so no raw Kaggle data is published.
It imports the old package name `creditRisk`, so it no longer runs as-is. See `legacy/README.md`.

In [ ]:
import os
import kagglehub
path = kagglehub.competition_download('home-credit-default-risk')

In [ ]:
for filename in os.listdir(path):
    print(filename)

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(path, 'application_train.csv'), encoding='ISO-8859-1')
for i, col in enumerate(df.columns):
    if df[col].isna().sum() == 0:
        print(i, col, df[col].dtype)

In [ ]:
df = pd.read_csv(os.path.join(path, 'bureau.csv'), encoding='ISO-8859-1')
for i, col in enumerate(df.columns):
    if df[col].isna().sum() == 0:
        print(i, col, df[col].dtype)

In [ ]:
pd.read_csv(os.path.join(path, 'installments_payments.csv'), encoding='ISO-8859-1')

In [ ]:
import duckdb
con = duckdb.connect("risk.duckdb")

csv_path = os.path.join(path, 'application_train.csv')

# con.sql(f"""
# CREATE TABLE application AS
# SELECT * FROM read_csv_auto('{csv_path}')
# """)

print(con.sql("SELECT COUNT(*) FROM application"))
print(con.sql("SELECT SK_ID_CURR, TARGET, AMT_INCOME_TOTAL, AMT_CREDIT FROM application LIMIT 5"))

In [ ]:
print(con.sql("""
SELECT TARGET, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM application
GROUP BY TARGET
"""))
# class imbalance: 8% of the applicants are defaulters (TARGET=1) and 92% are non-defaulters (TARGET=0)

In [ ]:
print(con.sql("SELECT COUNT(*) FROM bureau"))
print(con.sql("SELECT SK_ID_CURR, SK_ID_BUREAU, CREDIT_ACTIVE FROM bureau LIMIT 5"))

In [ ]:
print(con.sql("""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT SK_ID_BUREAU) AS unique_bureau_ids,
  COUNT(DISTINCT SK_ID_CURR) AS unique_customers
FROM bureau
"""))

In [ ]:
print(con.sql("""
SELECT
  COUNT(*) AS applicants,
  COUNT(b.SK_ID_CURR) AS with_bureau_history
FROM application a
LEFT JOIN (SELECT DISTINCT SK_ID_CURR FROM bureau) b
  USING (SK_ID_CURR)
"""))

In [ ]:
print(con.sql("""
SELECT SK_ID_CURR, SK_ID_BUREAU, CREDIT_ACTIVE, AMT_CREDIT_SUM
FROM bureau
WHERE SK_ID_CURR = 215354
"""))

In [ ]:
print(con.sql("""
SELECT
  SK_ID_CURR,
  COUNT(*) AS n_loans,
  AVG(AMT_CREDIT_SUM) AS avg_amount
FROM bureau
WHERE SK_ID_CURR = 215354
GROUP BY SK_ID_CURR
"""))
# Binder Error: column "SK_ID_CURR" must appear in the GROUP BY clause or must be part of an aggregate function.
# Either add it to the GROUP BY list, or use ANY_VALUE("SK_ID_CURR") if the exact value of "SK_ID_CURR" is not important.

In [ ]:
print(con.sql("""
SELECT
  SK_ID_CURR,
  COUNT(*) AS n_loans,
  AVG(AMT_CREDIT_SUM) AS avg_amount
FROM bureau
GROUP BY SK_ID_CURR
LIMIT 5
"""))

In [ ]:
con.sql("""
CREATE TABLE bureau_agg AS
SELECT
  SK_ID_CURR,
  COUNT(*) AS n_loans,
  AVG(AMT_CREDIT_SUM) AS avg_amount
FROM bureau
GROUP BY SK_ID_CURR
""")

In [ ]:
con.sql("""
SELECT 
    SK_ID_CURR, 
    TARGET, 
    COALESCE(n_loans, 0) AS n_loans, 
    avg_amount
FROM application a
LEFT JOIN bureau_agg b USING (SK_ID_CURR)
LIMIT 10
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE bureau_agg AS
SELECT
    SK_ID_CURR,
    COUNT(*) AS n_loans,
    AVG(AMT_CREDIT_SUM) AS avg_amount,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END) AS n_active_loans
FROM bureau
GROUP BY SK_ID_CURR
""")

In [ ]:
con.sql("""
SELECT
    SK_ID_CURR,
    n_loans,
    avg_amount,
    n_active_loans
FROM bureau_agg
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE risk_mart AS
SELECT
    SK_ID_CURR, 
    TARGET,
    COALESCE(n_loans, 0) AS n_loans, 
    avg_amount,
    n_active_loans
FROM application a
LEFT JOIN bureau_agg b USING (SK_ID_CURR)
""")

In [ ]:
print(con.sql("""
SELECT
  n_active_loans,
  COUNT(*) AS customers,
  ROUND(AVG(TARGET) * 100, 1) AS default_rate_pct
FROM risk_mart
GROUP BY n_active_loans
ORDER BY n_active_loans
LIMIT 8
"""))

## Data Quality (DQ)
A DQ check is a SQL query that counts the rows breaking a rule: 0 means pass, anything else is a problem.

In [ ]:
# Uniqueness check: Are there any duplicate SK_ID_CURR values in the risk_mart table?
print(con.sql("""
SELECT COUNT(*) - COUNT(DISTINCT SK_ID_CURR) AS duplicate_ids
FROM risk_mart
"""))

In [ ]:
# Completeness check: Are there any NULL values in the risk_mart table?
con.sql("""
SELECT COUNT(*) AS incomplete_rows
FROM risk_mart
WHERE SK_ID_CURR IS NULL OR TARGET IS NULL
""")

In [ ]:
# Validity check
print(con.sql("""
SELECT MIN(DAYS_EMPLOYED), MAX(DAYS_EMPLOYED)
FROM application
"""))

In [ ]:
con.sql("""
SELECT 
    NAME_INCOME_TYPE,
    COUNT(*) AS num_rows
FROM application
WHERE DAYS_EMPLOYED = 365243
GROUP BY NAME_INCOME_TYPE
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE risk_mart AS
SELECT
    SK_ID_CURR, 
    TARGET,
    COALESCE(n_loans, 0) AS n_loans, 
    COALESCE(n_active_loans, 0) AS n_active_loans,
    avg_amount,
    CASE WHEN DAYS_EMPLOYED = 365243 THEN NULL ELSE DAYS_EMPLOYED END AS days_employed,
    CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END AS is_not_employed
FROM application a
LEFT JOIN bureau_agg b USING (SK_ID_CURR)
""")

In [ ]:
print(con.sql("""
SELECT
  SUM(is_not_employed) AS not_employed,
  MAX(days_employed) AS max_days,
  SUM(CASE WHEN n_active_loans IS NULL THEN 1 ELSE 0 END) AS null_active
FROM risk_mart
"""))

print(con.sql("SELECT COUNT(*) FROM risk_mart WHERE days_employed = 0"))

In [ ]:
from creditRisk.DQ import data_quality_checks
data_quality_checks(con)

In [ ]:
# Consistency check
con.sql("""
SELECT 
    (SELECT COUNT(*) FROM risk_mart) - (SELECT COUNT(*) FROM application) AS row_count_difference
""")

## Data Lineage

In [ ]:
lineage = [
    # (source table.column,       target table.column,          transformation)
    ("bureau.CREDIT_ACTIVE",      "bureau_agg.n_active_loans",  "SUM(CASE WHEN CREDIT_ACTIVE='Active' THEN 1 ELSE 0 END)"),
    ("bureau_agg.n_active_loans", "risk_mart.n_active_loans",   "COALESCE(n_active_loans, 0)"),
    ("application.TARGET",        "risk_mart.TARGET",           "direct copy"),
    ("bureau.SK_ID_BUREAU", "bureau_agg.n_loans", "COUNT(*) GROUP BY SK_ID_CURR"),
    ("bureau_agg.n_loans", "risk_mart.n_loans", "COALESCE(n_loans, 0)"),
    ("bureau.AMT_CREDIT_SUM", "bureau_agg.avg_amount", "AVG(AMT_CREDIT_SUM) GROUP BY SK_ID_CURR"),
    ("bureau_agg.avg_amount", "risk_mart.avg_amount", "direct copy"),
    ("application.DAYS_EMPLOYED", "risk_mart.days_employed", "CASE WHEN DAYS_EMPLOYED = 365243 THEN NULL ELSE DAYS_EMPLOYED END"),
    ("application.DAYS_EMPLOYED", "risk_mart.is_not_employed", "CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END"),
    ("application.SK_ID_CURR", "risk_mart.SK_ID_CURR", "direct copy"),
    # Debt-to-income ratio: one target, TWO sources
    ("application.AMT_CREDIT",       "risk_mart.credit_income_ratio", "AMT_CREDIT / AMT_INCOME_TOTAL"),
    ("application.AMT_INCOME_TOTAL", "risk_mart.credit_income_ratio", "AMT_CREDIT / AMT_INCOME_TOTAL"),
]
lineage

In [ ]:
def trace(column):
    # e.g. trace("risk_mart.n_active_loans") should print:
    # risk_mart.n_active_loans
    #   <- bureau_agg.n_active_loans  [COALESCE(n_active_loans, 0)]
    #   <- bureau.CREDIT_ACTIVE       [SUM(CASE WHEN ...)]
    target = column
    while target in [t[1] for t in lineage]:
        source = [t[0] for t in lineage if t[1] == target][0]
        transformation = [t[2] for t in lineage if t[1] == target][0]
        print(f"{target} <- {source} [{transformation}]")
        target = source

In [ ]:
trace("risk_mart.n_active_loans")

In [ ]:
def trace(column, depth=0):
    # find ALL rows whose target is this column (not just [0])
    for source, target, transform in lineage:
        if target == column:
            print("  " * depth + f"<- {source} [{transform}]")
            trace(source, depth + 1)

In [ ]:
trace("risk_mart.credit_income_ratio")

In [ ]:
from graphviz import Digraph
from creditRisk.config import Config
config = Config()

def draw_lineage(lineage):
    g = Digraph(graph_attr={"rankdir": "LR"})  # left to right
    for source, target, transform in lineage:
        g.edge(source, target)
    return g

g = draw_lineage(lineage)  # last line in a cell renders the graph inline
g.render(filename=config.OUTPUTS_DIR / "lineage", format="png", cleanup=True)

In [ ]:
# lineage says these columns exist in risk_mart
documented = {t.split(".")[1] for _, t, _ in lineage if t.startswith("risk_mart.")}

# columns that actually exist in the table
actual = set(con.sql("SELECT * FROM risk_mart LIMIT 0").columns)

print("In lineage but not in table:", documented - actual)
print("In table but not in lineage:", actual - documented)

In [ ]:
print(con.sql("""
SELECT CREDIT_CURRENCY, COUNT(*) AS n
FROM bureau
GROUP BY CREDIT_CURRENCY
"""))

In [ ]:
con.close()